# Prediksi Persetujuan Kredit Bank — Skor Kredit Hampir Menentukan Semuanya
### Predictive Analytics — Klasifikasi dengan Decision Tree, Random Forest, dan Logistic Regression
**Kelompok 5 · Program Studi Teknik Informatika · STMIK PONTIANAK**

Notebook ini mendampingi dokumen materi dan slide presentasi dengan judul yang sama.
Semua angka di materi, slide, dan tanya-jawab berasal dari notebook ini dan bisa diperiksa ulang sel demi sel.

**Pertanyaan bisnis:** dari data seorang pemohon kredit, bisakah bank menebak apakah pengajuannya akan disetujui atau ditolak — dan faktor apa yang sebenarnya menentukan?

**Temuan utama (jangan dilewati):** model terbaik mencapai akurasi 98,13%, tapi itu BUKAN kabar baik. Tanpa skor CIBIL, semua fitur lain bersama-sama tidak bisa memprediksi apa-apa (akurasi di bawah tebakan asal). Dataset ini praktis adalah aturan ambang skor CIBIL yang menyamar sebagai masalah machine learning. Notebook ini membuktikannya dengan tiga skenario: dengan CIBIL, tanpa CIBIL, dan hanya CIBIL.


## 1. Ambil data
Dataset: Loan Approval Prediction Dataset (Kaggle, Archit Sharma).
File CSV ikut tersimpan di repo yang sama dengan notebook ini, jadi Colab cukup mengunduhnya dari GitHub.
Kalau unduhan gagal (misalnya tidak ada internet), unggah file `loan-approval-4269.csv` secara manual ke Colab, lalu jalankan ulang sel ini.


In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/awangsuryarmdhn/weight-loss-regression/main/loan-approval-4269.csv"
try:
    df = pd.read_csv(url)
    print("Data terunduh dari GitHub.")
except Exception as e:
    print("Unduhan gagal:", e)
    print("Unggah file loan-approval-4269.csv ke Colab, lalu jalankan: df = pd.read_csv('loan-approval-4269.csv')")
    raise
# Nama kolom dan nilai di CSV asli ada spasi di depan — dibersihkan dulu.
df.columns = [c.strip() for c in df.columns]
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()
print(df.shape)  # 4.269 baris, 13 kolom
print(df.columns.tolist())
print("Data kosong:", int(df.isna().sum().sum()))
df.head()


## 2. Kenal datanya dulu
Targetnya `loan_status`: Approved (disetujui) atau Rejected (ditolak). Kolomnya:

| Kolom | Arti | Contoh |
|---|---|---|
| no_of_dependents | Jumlah tanggungan | 2 |
| education | Pendidikan (Graduate / Not Graduate) | Graduate |
| self_employed | Wiraswasta atau bukan | No |
| income_annum | Penghasilan setahun | 5.000.000 |
| loan_amount | Jumlah pinjaman yang diminta | 15.000.000 |
| loan_term | Jangka waktu pinjaman (bulan) | 12 |
| cibil_score | Skor kredit CIBIL (300–900) | 703 |
| residential_assets_value | Nilai aset rumah tinggal | 2.400.000 |
| commercial_assets_value | Nilai aset komersial | 2.000.000 |
| luxury_assets_value | Nilai aset mewah | 8.000.000 |
| bank_asset_value | Nilai aset di bank | 3.000.000 |
| loan_status | Hasil: Approved / Rejected | Approved |

Angka uang di dataset ini ditulis tanpa satuan mata uang yang dijelaskan sumbernya. Di materi dan presentasi, angka ditulis dalam rupiah seperti di file aslinya — yang penting adalah perbandingannya, bukan nilai tukar.


In [ ]:
print(df["loan_status"].value_counts())
print(df["loan_status"].value_counts(normalize=True).round(4))
print("\nBaseline (selalu tebak Approved):", round((df["loan_status"] == "Approved").mean() * 100, 2), "%")
print(df.describe().round(1).to_string())


## 3. EDA: siapa yang disetujui?
Tiga hal dicek sebelum membuat model: berapa persen yang disetujui, apakah penghasilan membedakan, dan bagaimana skor CIBIL membedakan.


In [ ]:
print("Persetujuan menurut pendidikan:")
print(df.groupby("education")["loan_status"].apply(lambda s: (s == "Approved").mean() * 100).round(2))
print("\nPersetujuan menurut status wiraswasta:")
print(df.groupby("self_employed")["loan_status"].apply(lambda s: (s == "Approved").mean() * 100).round(2))
print("\nPenghasilan (median) menurut hasil:")
print(df.groupby("loan_status")["income_annum"].median())
print("\nJumlah pinjaman (median) menurut hasil:")
print(df.groupby("loan_status")["loan_amount"].median())
print("\nSkor CIBIL menurut hasil:")
print(df.groupby("loan_status")["cibil_score"].describe().round(1))


In [ ]:
print("Persetujuan menurut ambang skor CIBIL:")
for t in [500, 600, 700]:
    hi = df[df["cibil_score"] >= t]
    lo = df[df["cibil_score"] < t]
    print(f"  cibil >= {t}: {len(hi)} orang, disetujui {(hi['loan_status'] == 'Approved').mean() * 100:.1f}%")
    print(f"  cibil <  {t}: {len(lo)} orang, disetujui {(lo['loan_status'] == 'Approved').mean() * 100:.1f}%")
print("\nPersetujuan menurut kelompok skor CIBIL:")
df["cibil_band"] = pd.cut(df["cibil_score"], [0, 400, 500, 600, 700, 800, 901])
print(df.groupby("cibil_band", observed=True)["loan_status"].apply(lambda s: round((s == "Approved").mean() * 100, 1)))
print(df.groupby("cibil_band", observed=True).size())


**Yang terlihat di sini:**
- Penghasilan hampir tidak membedakan: median penghasilan yang disetujui 5.000.000, yang ditolak 5.100.000.
- Jumlah pinjaman juga hampir sama: median 14,6 juta (disetujui) vs 14,5 juta (ditolak).
- Pendidikan dan status wiraswasta juga hampir sama: lulusan 62,45% disetujui vs bukan lulusan 61,98%; wiraswasta 62,23% vs bukan 62,20%.
- Skor CIBIL yang membedakan dengan tajam: yang disetujui rata-rata 703, yang ditolak rata-rata 429. Skor 600 ke atas disetujui 99,5%, di bawah 600 hanya 24,9%.


## 4. Siapkan model
Tiga model dibandingkan: Decision Tree (kedalaman maksimal 5, supaya masih bisa dijelaskan), Random Forest (200 pohon), dan Logistic Regression.
Data dibagi 80% latih dan 20% uji, dengan stratifikasi supaya proporsi disetujui/ditolak sama di kedua bagian. Seed 42 supaya hasilnya bisa diulang.
Tiga skenario dijalankan pada pembagian yang sama:
1. **Dengan CIBIL** — semua fitur dipakai.
2. **Tanpa CIBIL** — semua fitur kecuali skor CIBIL.
3. **Hanya CIBIL** — satu-satunya fitur adalah skor CIBIL.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

y = (df["loan_status"] == "Approved").astype(int)
X = df.drop(columns=["loan_id", "loan_status", "cibil_band"])

def jalankan(X_, label):
    Xtr, Xte, ytr, yte = train_test_split(X_, y, test_size=0.2, random_state=42, stratify=y)
    num = X_.select_dtypes(include=[np.number]).columns.tolist()
    cat = [c for c in X_.columns if c not in num]
    pre = ColumnTransformer([("num", "passthrough", num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    hasil = {}
    for nama, model in [
        ("Decision Tree", DecisionTreeClassifier(random_state=42, max_depth=5)),
        ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ]:
        pipe = Pipeline([("pre", pre), ("m", model)])
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        prob = pipe.predict_proba(Xte)[:, 1]
        hasil[nama] = (accuracy_score(yte, pred), roc_auc_score(yte, prob), confusion_matrix(yte, pred))
        print(f"{label} | {nama}: akurasi={hasil[nama][0] * 100:.2f}%  AUC={hasil[nama][1]:.3f}")
        print(f"   confusion matrix (baris = sebenarnya Ditolak/Disetujui, kolom = ditebak Ditolak/Disetujui):\n{hasil[nama][2]}")
    return hasil

print("Baseline (selalu tebak Disetujui): 62,22%\n")
print("=== Skenario 1: DENGAN skor CIBIL ===")
h_dengan = jalankan(X, "DENGAN CIBIL")
print("\n=== Skenario 2: TANPA skor CIBIL ===")
h_tanpa = jalankan(X.drop(columns=["cibil_score"]), "TANPA CIBIL")
print("\n=== Skenario 3: HANYA skor CIBIL ===")
h_hanya = jalankan(X[["cibil_score"]], "HANYA CIBIL")


## 5. Faktor apa yang paling penting?
Random Forest menghitung seberapa besar tiap fitur dipakai untuk memisahkan pemohon yang disetujui dan ditolak.


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num = X.select_dtypes(include=[np.number]).columns.tolist()
cat = [c for c in X.columns if c not in num]
pre = ColumnTransformer([("num", "passthrough", num),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
rf = Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
rf.fit(Xtr, ytr)
nama_fitur = pre.fit(Xtr).get_feature_names_out()
penting = sorted(zip(nama_fitur, rf.named_steps["m"].feature_importances_), key=lambda z: -z[1])
for f, v in penting:
    print(f"{f.replace('num__', '').replace('cat__', '')}: {v * 100:.1f}%")


## 6. Demo: dua pemohon yang sama, skor CIBIL berbeda
Dua profil ini identik di semua hal — penghasilan 5 juta setahun, pinjaman 15 juta, jangka 12 bulan, lulusan, bukan wiraswasta, 2 tanggungan, aset yang sama. Yang beda hanya skor CIBIL: 780 vs 420.
Model yang dipakai adalah Random Forest dari skenario 1 (dengan CIBIL).


In [ ]:
profil = {"no_of_dependents": 2, "education": "Graduate", "self_employed": "No",
          "income_annum": 5000000, "loan_amount": 15000000, "loan_term": 12,
          "residential_assets_value": 2400000, "commercial_assets_value": 2000000,
          "luxury_assets_value": 8000000, "bank_asset_value": 3000000}
for cib in [780, 420]:
    p = dict(profil); p["cibil_score"] = cib
    prob = rf.predict_proba(pd.DataFrame([p]))[0, 1] * 100
    print(f"Skor CIBIL {cib}: peluang disetujui {prob:.1f}% -> {'DISETUJUI' if prob >= 50 else 'DITOLAK'}")


## 7. Kesimpulan
1. **Model terlihat sangat akurat — 98,13% untuk Random Forest — tapi itu menyesatkan.** Satu fitur saja, skor CIBIL, sudah memberi akurasi 96,60% untuk Decision Tree. Model ini praktis hanya meniru aturan "skor CIBIL 600 ke atas hampir selalu disetujui".
2. **Tanpa skor CIBIL, model gagal.** Akurasi Decision Tree 60,66% dan Random Forest 58,67% — lebih buruk daripada sekadar selalu menebak "disetujui" (62,22%). Semua fitur lain bersama-sama — penghasilan, jumlah pinjaman, pendidikan, aset — tidak bisa memprediksi persetujuan.
3. **Penghasilan tidak menentukan.** Median penghasilan yang disetujui dan ditolak hampir sama (5,0 juta vs 5,1 juta). Yang menentukan adalah riwayat kredit yang tercermin di skor CIBIL.
4. **Untuk bank:** skor CIBIL cocok untuk triase cepat, tapi keputusan otomatis hanya dari skor berisiko tidak adil — skor bisa salah, dan pemohon tanpa riwayat kredit (misalnya baru pertama kali meminjam) akan selalu kalah.
5. **Untuk nasabah:** menjaga skor kredit — bayar tepat waktu, jangan menunggak — jauh lebih berpengaruh daripada menaikkan penghasilan sedikit.

**Keterbatasan:** dataset ini tidak mencatat kota, pekerjaan detail, atau alasan penolakan bank yang sebenarnya. Asal datanya tidak dinyatakan sebagai data bank Indonesia, jadi jangan diklaim sebagai gambaran bank Indonesia. Model ini alat belajar, bukan alat untuk memutuskan kredit sungguhan.
